In [ ]:
# ==========================================
# 阶段一：TCN 数据管线 - 按年分块清洗与落盘
# ==========================================
import pandas as pd
import numpy as np
import torch
import os
import gc
from jqdata import *
import warnings
warnings.filterwarnings('ignore')

# --- 数据提取参数 ---
START_YEAR = 2018
END_YEAR = 2023
SEQ_LENGTH = 30           # 历史观察窗口 (30天)
FUTURE_DAYS = 5           # 预测未来收益窗口 (5天)
FEATURE_COLS = ['open', 'close', 'high', 'low', 'volume', 'money']

def process_and_save_yearly_data(year):
    print(f"🚀 开始处理 {year} 年的切片数据...")
    
    # 获取该年所有交易日
    tdays = get_trade_days(start_date=f'{year}-01-01', end_date=f'{year}-12-31')
    # 每隔 5 个交易日采样一次截面（平衡数据量与相关性，相当于周频采样）
    sample_days = tdays[::5]
    
    X_year, y_year = [], []
    valid_samples = 0
    
    for date in sample_days:
        # ---------------------------------------------------------
        # 1. 绝对对齐实盘：提取当天的目标选股池 (ROE>5, ROA>3, 市值最小150)
        # ---------------------------------------------------------
        q = query(valuation.code).filter(
            indicator.roe > 5.0,
            indicator.roa > 3.0
        ).order_by(valuation.market_cap.asc()).limit(150)
        
        try:
            df_fund = get_fundamentals(q, date=date)
            pool = list(df_fund['code'])
        except:
            continue
            
        if not pool: continue

        # ---------------------------------------------------------
        # 2. 获取历史量价张量 (过去 30 天)
        # ---------------------------------------------------------
        hist_df = get_price(pool, end_date=date, count=SEQ_LENGTH, 
                            fields=FEATURE_COLS, frequency='daily', panel=False)
        if hist_df.empty: continue
        
                # ---------------------------------------------------------
        # 3. 获取未来绝对收益 (未来 5 天) - 【已修复 API 参数冲突】
        # ---------------------------------------------------------
        all_trade_days = list(get_all_trade_days())
        
        # 获取当前日期在全局日历中的索引
        current_idx = all_trade_days.index(date)
        
        # 边缘保护：如果是年底最后几天，未来不足 5 天，则直接跳过该截面
        if current_idx + FUTURE_DAYS >= len(all_trade_days):
            continue
            
        future_date = all_trade_days[current_idx + FUTURE_DAYS]

        # 【核心修复】：使用 start_date + end_date 的合法组合
        fut_df = get_price(pool, start_date=date, end_date=future_date, 
                           fields=['close'], frequency='daily', panel=False)
                           
        if fut_df.empty: continue

        # ---------------------------------------------------------
        # 4. 截面打标与张量构造
        # ---------------------------------------------------------
        future_rets = {}
        hist_tensors = {}
        
        for stock in pool:
            # 提取历史特征
            s_hist = hist_df[hist_df['code'] == stock]
            if len(s_hist) < SEQ_LENGTH: continue # 数据长度不够则剔除
            
            # 特征 Z-Score 标准化 (在 30 天时间轴上局部标准化，绝对无未来函数)
            data_mat = s_hist[FEATURE_COLS].values
            mean = np.mean(data_mat, axis=0, keepdims=True)
            std = np.std(data_mat, axis=0, keepdims=True) + 1e-8
            norm_mat = (data_mat - mean) / std
            hist_tensors[stock] = norm_mat
            
            # 计算未来收益
            s_fut = fut_df[fut_df['code'] == stock]
            if len(s_fut) < FUTURE_DAYS + 1: continue
            ret = s_fut.iloc[-1]['close'] / s_fut.iloc[0]['close'] - 1
            future_rets[stock] = ret
            
        if not future_rets: continue
        
        # 将收益转化为截面二分类标签 (排名前 30% 标为 1，其余为 0)
        ret_series = pd.Series(future_rets)
        labels = (ret_series.rank(pct=True) >= 0.70).astype(float)
        
        # 组装当天的有效样本
        for stock in labels.index:
            X_year.append(hist_tensors[stock])
            y_year.append(labels[stock])
            valid_samples += 1

    if valid_samples > 0:
        # 转换并转置维度以适配 PyTorch Conv1d: (Batch, Features, Seq)
        X_np = np.transpose(np.array(X_year), (0, 2, 1))
        y_np = np.array(y_year).reshape(-1, 1)
        
        X_tensor = torch.tensor(X_np, dtype=torch.float32)
        y_tensor = torch.tensor(y_np, dtype=torch.float32)
        
        # 落盘保存为 .pt 文件
        save_path = f'tcn_chunk_{year}.pt'
        torch.save((X_tensor, y_tensor), save_path)
        print(f"✅ {year} 年处理完毕! 提取纯正样本 {valid_samples} 个，已落盘至 {save_path}")
    
    # 强制清理内存
    del X_year, y_year
    gc.collect()

# 执行多年数据提取
for y in range(START_YEAR, END_YEAR + 1):
    process_and_save_yearly_data(y)
    
print("🎉 所有年份数据清洗落盘完毕！请进行第二部分模型训练。")

